In [ ]:


import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import random
import os
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error
import shap
import matplotlib.animation as animation
from sklearn.decomposition import PCA
from sklearn.linear_model import Ridge
from sklearn.linear_model import LinearRegression
from sklearn.neural_network import MLPRegressor
import plotly.graph_objects as go
import plotly.io as pio
pio.renderers.default = "browser"

import plotly.graph_objects as go
from plotly.subplots import make_subplots
from numpy.linalg import inv
from sklearn.decomposition import PCA
import numpy as np

import seaborn as sns

import shap
from numpy.linalg import inv


#!pip install seaborn
#!pip install uproot

import uproot

from numpy.linalg import inv
from plotly.subplots import make_subplots
from sklearn.decomposition import PCA

file_path = r"C:\Users\lenao\Project_emulator_python\inputs\fdsFit_Martini1pi_2021.root"
root_file = uproot.open(file_path)

systematics = [
    'Flux Systematics',
    'ND280 Detector Systematics',
    'Cross-Section Systematics',
    'Cross-Section (binned) Systematics'
]

samples = ['FHC FGD1 #nu_{#mu} CC 0#pi 0p 0#gamma', 'FHC FGD1 #nu_{#mu} CC 0#pi Np 0#gamma']
variables = ['Pmu', 'CosThetamu']
reactions = ['1;1', '2;1']

predictions = {}

#one simply extract the postfit prediction in momentum and direction from the original fitter output
for sample in samples:
    predictions[sample] = {}

    for variable in variables:
        predictions[sample][variable] = {}

        base_path = f"FitterEngine/postFit/samples/histograms/{sample}/{variable}/ReactionCode"

        try:
            available_reactions = root_file[base_path].keys()
        except:
            print(f"Missing path: {base_path}")
            continue

        for reaction in reactions:

            try:
                hist = root_file[f"{base_path}/{reaction}/MC_TH1D"]

                values = hist.values()
                edges = hist.axes[0].edges()

                predictions[sample][variable][reaction] = {
                    "values": np.array(values), # number of events per bin
                    "edges": edges # pmu/cos edges
                }

                y = predictions[sample][variable][reaction]["values"]
                edges = predictions[sample][variable][reaction]["edges"]

                plt.figure(figsize=(6,4))

                plt.step(edges[:-1], y, where="post", label="prediction")
                plt.bar(edges[:-1], y, width=np.diff(edges), alpha=0.3, align="edge")

                plt.xlabel("Bin")
                plt.ylabel("Event number")
                plt.title(f"{sample} - {variable} - {reaction}")
                plt.grid()
                plt.legend()
                plt.show()


            except Exception as e:
                print(f"Erreur {reaction}: {e}")

########################################################################################
                

# A short function to be able to read the right label names in my file, no tricky but you might want to check yours :)
def normalize(name):
    
    if "ND280" in name:
        return name.replace("ND280 Detector Systematics/","")
    if "Flux" in name:
        return name.replace("Flux Systematics/","")
    if "Cross-Section (binned) Systematics" in name:
        return name.replace("Cross-Section (binned) Systematics/","")
    if "Cross-Section Systematics" in name:
        return name.replace("Cross-Section Systematics/","")
    return name

##########################################################################################

bestfit_dict = {}
sigma_dict = {}

for sys_name in systematics:

    path = f"FitterEngine/postFit/Hesse/errors/{sys_name}/values/postFitErrors_TH1D"
    scan = root_file[path]

    values = np.array(scan.values()) # central values ~ bestfit values
    errors = None

    try:
        errors = np.array(scan.errors())
    except:
        errors = np.sqrt(scan.variances())

    labels = scan.axes[0].labels()

    print(f"{sys_name} -> {len(labels)} params")

    for name, val, err in zip(labels, values, errors):

        n = normalize(name)
        key = (sys_name, n)

        bestfit_dict[key] = val # bestfit values
        sigma_dict[key] = err # uncertainties

# Next, we will need to extract the covariance matrix stocked in the root file or yours
cov_obj = root_file[
    "FitterEngine/postFit/Hesse/hessian/postfitCovarianceOriginal_TH2D"# cov matrix without PCA in first place
]

cov = cov_obj.values()

labels = list(cov_obj.axes[0].labels())
assert labels == list(cov_obj.axes[1].labels())
cov_labels = [normalize(n) for n in labels]
print(cov_labels)


# This is important to be sure all the different labels and parameters names are matching ! if not, ERROR !!
mean = []
indices = []
matched_keys = []

bestfit_lookup = {} #a dictionnary to keep same parameters as in the cov matrix, so they match

for k in bestfit_dict:
    bestfit_lookup[k[1]] = k # we make correspond the name of the parameter to its quantity, so k[1]~ param. name, k~(quantity, param. name)

for i, cov_name in enumerate(cov_labels):

    if cov_name in bestfit_lookup: #names which are actually matching
        key = bestfit_lookup[cov_name]

        mean.append(bestfit_dict[key])# we rebuild in the same order
        indices.append(i)
        matched_keys.append(key)

mean = np.array(mean) #converted in numpy array
cov_reduced = cov[np.ix_(indices, indices)]

print("\nmean shape:", mean.shape)
print("cov shape:", cov_reduced.shape)

print("cov symmetric:", np.allclose(cov_reduced, cov_reduced.T))# check cov symmetry

eigvals = np.linalg.eigvals(cov_reduced)
print("cov positive definite:", np.all(eigvals >= 0))# check cov positive definite

#print("\nBESTFIT labels (raw):")
#for k in list(bestfit_dict.keys())[:20]:
   # print(repr(k[1]))


In [ ]:
def modify_covariance(cov, corr_strength=1.0, scale=1.0):# the corr strenght highlights the new correlation (1.0 = same corr, 0.0 = none or diag matrix)

    # corr matrix extraction from cov
    std = np.sqrt(np.diag(cov))
    corr = cov / np.outer(std, std)

    # modification of corr
    corr_mod = corr_strength * corr + (1 - corr_strength) * np.eye(len(cov))

    # reconstruction
    cov_mod = corr_mod * np.outer(std, std)

    # global scaling
    cov_mod *= scale

    return cov_mod


In [ ]:
#the heart of the script ! generation of pseudo-date keeping the cov matrix as you implement it or not, according to your studies
#you can modify the correlations and see their respective weight on the predictions at the end

def sample_theta(mean, cov, n):

    return np.random.multivariate_normal(mean, cov, size=n)
    

In [ ]:

def propagate_to_hist(y_nominal, theta, mean, n_bins, n_params):
    
    R = np.random.normal(0, 0.02, size=(n_bins, n_params))

    delta = (theta - mean)

    shift = R @ delta   

    return y_nominal * (1 + shift)

In [ ]:
print("mean shape:", mean.shape)
print("cov shape:", cov_reduced.shape)

cov = modify_covariance(cov_reduced, corr_strength=1.0, scale=1.0)
R_dict = {}

for s in samples:
    for v in variables:
        for r in reactions:

            key = (s, v, r)

            y = predictions[s][v][r]["values"]

            if key not in R_dict:
                n_bins = len(y)
                n_params = len(mean)
                R_dict[key] = np.random.normal(0, 0.02, size=(n_bins, n_params))

            R = R_dict[key]

            theta_samples = sample_theta(mean, cov, 200)
            print("theta_samples:",len(theta_samples))

            y_mouch = []

            for theta in theta_samples:
                #delta = theta - mean
                #print(delta)
                #shift = R @ delta
                #yy = np.abs(y * (1 + shift))
                yy = propagate_to_hist(y, theta, mean) 
                y_mouch.append(yy)

            y_mouch = np.array(y_mouch)
            print(len(y_mouch))

            Bins = np.arange(len(y))

            plt.figure(figsize=(8,5))

            for i in range(y_mouch.shape[0]):
                plt.scatter(Bins, y_mouch[i], color='blue', alpha=0.05)

            mean_bins = y_mouch.mean(axis=0)
            std_bins  = y_mouch.std(axis=0)

            plt.step(Bins, mean_bins, color='red', linewidth=2, label='mean pseudo data')
            plt.fill_between(Bins, mean_bins-std_bins, mean_bins+std_bins, color='red', alpha=0.3, label='±1σ')

            plt.step(Bins, y, color='black', linewidth=2, label='nominal')

            plt.title(f"{s} | {v} | {r}")
            plt.xlabel("bin")
            plt.ylabel("events")
            plt.legend()
            plt.grid(True)
            plt.show()


In [ ]:
param_names = np.array(cov_labels)

#want to divide plots by systematics, here are flux, xsec and detector
groups = {
    "flux": np.arange(0, 100),
    "det":  np.arange(100, 100 + 552),
    "xsec": np.arange(100+552, 100+551+56),
    "xsecEb": np.arange(100+551+56, 100+551+56+4),
}

def get_index_in_group(name, group, param_names, groups):

    idx_group = groups[group]              # indices globaux du groupe
    names_group = param_names[idx_group]   # labels du groupe
    matches = np.where(names_group == name)[0]

    if len(matches) == 0:
        raise ValueError(f"{name} not found in group {group}")

    return idx_group[matches[0]]

In [ ]:
for g, idxx in groups.items():
    print(g, ":", len(idxx))

print("Total parameters:", len(param_names))
!pip install umap-learn

datasets = {}
n_params = len(mean)

for s in samples:
    for v in variables:
        for r in reactions:
    
                print("\n==============================")
                print(s, v, r)
                print("==============================")
    
                y_nominal = predictions[s][v][r]["values"]
                n_bins = len(y_nominal)
                
                key = (s, v, r)
    
                if key not in R_dict:
                    R_dict[key] = np.random.normal(
                        0, 0.02,
                        size=(len(y_nominal), len(mean))
                    )
    
                R = R_dict[key]
    
                theta_samples = sample_theta(mean, cov, 200)
                X = np.array(theta_samples)
                Y = []
    
                for theta in X:
                    #delta = theta - mean
                    #shift = R @ delta
                    #yy = np.abs(y_nominal * (1 + shift))
                    yy = propagate_to_hist(y_nominal, theta, mean)
                    Y.append(yy)
    
                Y = np.array(Y)
    
                print("X:", X.shape, "Y:", Y.shape)
    
                Y_norm = Y/(y_nominal + 1e-8) - 1
    
                corr_sys = np.corrcoef(X, rowvar=False)    

                for g, idx in groups.items():
    
                    if len(idx) < 2:
                        continue
    
                    corr_sub = np.corrcoef(X[:, idx], rowvar=False)

                    print("ouuu")
    
                    plt.figure(figsize=(5,4))
                    sns.heatmap(
                        corr_sub,
                        cmap="coolwarm",
                        center=0,
                        xticklabels=[param_names[i] for i in idx],
                        yticklabels=[param_names[i] for i in idx]
                    )
                    plt.title(f"Sys-Sys {g}")
                    plt.xticks(rotation=90)
                    plt.yticks(rotation=0)
                    plt.tight_layout()
                    #plt.show()

                    print("ttt")
    
  
                corr_sys_bin = np.corrcoef(X.T, Y_norm.T)[:X.shape[1], X.shape[1]:]
    
                for g, idx in groups.items():
    
                    if len(idx) == 0:
                        continue
    
                    corr_sub = corr_sys_bin[idx, :]
    
                    plt.figure(figsize=(8,6))
                    sns.heatmap(
                        corr_sub,
                        cmap="coolwarm",
                        center=0,
                        yticklabels=[param_names[i] for i in idx]
                    )
                    plt.title(f"Sys → Bin {g}")
                    plt.xlabel("bins")
                    plt.ylabel("parameters")
                    plt.tight_layout()
                    plt.show()
                        
                param_names = np.array(param_names)  # IMPORTANT FIX numpy -> list
                EPS = 1e-8

                X_train, X_test, Y_train, Y_test = train_test_split(
                    X, Y_norm, test_size=0.2, random_state=42
                )
    

                modelsplusRF = {
                    "linear": LinearRegression(),
                    "ridge": Ridge(),
                    "mlp": MLPRegressor(hidden_layer_sizes=(64,64), max_iter=500),
                    "rf" : RandomForestRegressor(
                    n_estimators=300,
                    max_depth=None,
                    min_samples_leaf=5,
                    max_features=0.5,
                    bootstrap=True,
                    random_state=42,
                    n_jobs=-1
                    )
                }

                modelRF = {
                    "rf" : RandomForestRegressor(
                    n_estimators=300,
                    max_depth=None,
                    min_samples_leaf=5,
                    max_features=0.5,
                    bootstrap=True,
                    random_state=42,
                    n_jobs=-1
                    )
                }

                plt.figure(figsize=(8,5))

                plt.plot(Y_test[0], label="true", linewidth=3, color="black")
                
                for name, model in modelsplusRF.items():
                
                    model.fit(X_train, Y_train)
                    Y_pred = model.predict(X_test)
                
                    print(name, mean_squared_error(Y_test, Y_pred))
                    print("Y_test min/max:", Y_test.min(), Y_test.max())
                    print("Y_pred min/max:", Y_pred.min(), Y_pred.max())
                
                    # superposition
                    plt.plot(Y_pred[0], label=name)# first prediction
                
                plt.legend(loc="best")
                plt.title("Model comparison on histogram (per bin)")
                plt.grid(True)
                plt.show()

                modelRF = modelsplusRF["rf"]
                modelRF.fit(X_train, Y_train)
                
                Y_base = modelRF.predict(X_test)
                print("X_test shape:", X_test.shape)
                print("Y_base shape:", Y_base.shape)
                print("X_train shape:", X_train.shape)
                print("Y_train shape:", Y_train.shape)

                explainer = shap.TreeExplainer(model)
                shap_values = explainer.shap_values(X_test)
        
                bin_index = 5
        
                shap.summary_plot(
                    shap_values[:, :, bin_index],
                    X_test,
                    feature_names=param_names
                )
    
    
                pca = PCA(n_components=2)
                X_pca = pca.fit_transform(X)
    
                #plt.figure()
                #plt.scatter(X_pca[:,0], X_pca[:,1], alpha=0.3)
                #plt.title("PCA systematics")
                #plt.show()
    
                comp = pca.components_[0]
                idx = np.argsort(np.abs(comp))[::-1]
    
                print("\nTop PC1:")
                for i in idx[:10]:
                    print(param_names[i], comp[i])
    

                datasets[(s,v,r)] = (X_test, Y_base)
    
                ridge = Ridge()
                ridge.fit(X_train, Y_train)
                
                coef = ridge.coef_
                #plt.imshow(coef, aspect='auto', cmap='coolwarm')
                #plt.colorbar()
                #plt.title("Impact syst to bins (Ridge)")
                #plt.xlabel("paramètres")
                #plt.ylabel("bins")
                #plt.show()
    
    
                cov_inv = inv(cov)
                diff = X - mean
                dist = np.sqrt(np.einsum('ij,jk,ik->i', diff, cov_inv, diff))  # Distance de Mahalanobis
    
                #dist = (dist - dist.min()) / (dist.max() - dist.min() + 1e-12)
                
                # tri par distance
                idx_sort = np.argsort(dist)
                
                X_sorted = X[idx_sort]
                Y_sorted = Y_norm[idx_sort]
                X_pca = X_pca[idx_sort]
                dist = dist[idx_sort]
                
                n_bins = Y_base.shape[1]
                print("nbins ??",n_bins)
            
                bins = np.arange(n_bins)
                i = 0

                import networkx as nx                
                

                print("aiie")

                fig = go.Figure(data=[go.Scatter3d(
                x=X_pca[:,0],
                y=X_pca[:,1],
                z=dist,
                mode='markers',
                marker=dict(
                    size=3,
                    color=dist,
                    colorscale='Viridis'
                    )
                )])
                fig.update_layout(title="Systematics landscape (PCA + Mahalanobis)")
                #fig.show()

                print("booo")

                def compute_response(param_name):

                    param_names = np.array(cov_labels)

                    param_names = list(param_names)

                    p_idx = param_names.index(param_name)
                
                    X_var = X_test.copy()
                
                    delta = np.std(X_test[:, p_idx]) * 0.5
                    X_var[:, p_idx] += delta
                
                    Y_var = modelRF.predict(X_var)
                    Y_base = modelRF.predict(X_test)
                
                    Response = (Y_var - Y_base) / (Y_base + EPS)
            
                    return Response

                #fig.show()
                !pip install ipywidgets

                import numpy as np
                import plotly.graph_objects as go
                import ipywidgets as widgets
                from IPython.display import display
                
                
                # ============================================================
                # PARAMETER SELECTOR
                # ============================================================
                
                search_box = widgets.Text(
                placeholder='Search parameter...',
                description='Search:',
                layout=widgets.Layout(width='400px')
                )
                
                dropdown = widgets.Dropdown(
                    options=param_names,
                    description='Param:',
                    layout=widgets.Layout(width='700px')
                )
                
                def on_search_change(change):
                
                    text = change["new"].lower()
                
                    filtered = [p for p in param_names if text in p.lower()]
                
                    if len(filtered) == 0:
                        filtered = ["NO MATCH"]
                
                    dropdown.options = filtered
                
                search_box.observe(on_search_change, names='value')
                
                # ============================================================
                # 3D PLOT
                # ============================================================
                
                def plot_3d(param_name):

                    param_names = np.array(cov_labels)
                    
                    param_names = list(param_names)


                    p_idx = param_names.index(param_name)

                    X_var = X_test.copy()
                
                    delta = np.std(X_test[:, p_idx]) * 0.5
                    X_var[:, p_idx] += delta
                
                    Y_var = modelRF.predict(X_var)
                    Y_base = modelRF.predict(X_test)

                    Z = (Y_var - Y_base) / (Y_base + EPS)
                
                    n_predictions = Z.shape[0]
                    print("n_pred:", n_predictions)
                    n_bins = Z.shape[1]
                    print("nbins:",n_bins)
                    
                
                    predictions = np.arange(n_predictions)
                    bins = np.arange(n_bins)
                
                    P, B = np.meshgrid(predictions, bins, indexing='ij')
                
                    zmax = np.max(np.abs(Z))
                
                    fig = go.Figure()
                
                    fig.add_trace(go.Surface(
                        x=B,
                        y=P,
                        z=Z,
                        colorscale='RdBu',
                        reversescale=True,
                        cmin=-zmax,
                        cmax=zmax,
                        opacity=0.95,
                        colorbar=dict(title="ΔY/Y (ML impact)")
                    ))
                
                    fig.add_trace(go.Surface(
                        x=B,
                        y=P,
                        z=np.zeros_like(Z),
                        opacity=0.2,
                        showscale=False
                    ))
                
                    fig.update_layout(
                        title=f"ML Impact Surface : {param_name}",
                        width=1500,
                        height=900,
                        template='plotly_dark',
                        scene=dict(
                            xaxis_title="Bins",
                            yaxis_title="Predictions",
                            zaxis_title="Impact",
                            camera=dict(eye=dict(x=1.8, y=1.5, z=1.2))
                        )
                    )
                
                    fig.show(renderer="browser")
                
                # ============================================================
                # INTERACTIVE UI
                # ============================================================
                
                ui = widgets.VBox([
                    search_box,
                    dropdown
                ])
                
                out = widgets.interactive_output(
                    plot_3d,
                    {"param_name": dropdown}
                )
                
                display(ui, out)

                import plotly.graph_objects as go
                import plotly.express as px
                from sklearn.linear_model import Ridge
               
                Zdef = compute_response("#48_RES_Eb_O_numubar")
                pca = PCA(n_components=0.95)
                Response_reduced = pca.fit_transform(Zdef)
            
                
                import umap

                reducer = umap.UMAP(
                    n_neighbors=10,
                    min_dist=0.15,
                    metric='cosine',#cosine similarity matrix
                    random_state=42
                )
                
                embedding = reducer.fit_transform(Response_reduced)
                x = embedding[:,0]
                y = embedding[:,1]

                x = (x - x.min()) / (x.max() - x.min() + eps)
                y = (y - y.min()) / (y.max() - y.min() + eps)

                embedding.shape
                energy = np.sqrt(
                    np.sum(Zdef**2, axis=1)
                )

                signed_response = np.mean(
                    Z,
                    axis=1
                )

                

                fig = make_subplots(
                rows=3, cols=1,
                subplot_titles=(
                    "Param space (PCA)",
                    "Relative variation ΔY/Y",
                    "Distance to bestfit"
                    )
                )
                
                frames_main = []
                
                for i in range(1, len(X_sorted)):
                
                    y_current = Y_sorted[i]
                
                    frame = go.Frame(data=[
                
                        # PCA
                        go.Scatter(
                            x=X_pca[:i, 0],
                            y=X_pca[:i, 1],
                            mode="markers",
                            marker=dict(
                                size=6,
                                color=dist[:i],
                                colorscale="Viridis",
                                showscale=False
                            )
                        ),
                
                        # ΔY/Y
                        go.Scatter(
                            x=bins,
                            y=y_current,
                            mode="lines",
                            line=dict(color="green")
                        ),
                
                        # distance
                        go.Scatter(
                            y=dist[:i],
                            mode="lines",
                            line=dict(color="black")
                        )
                
                    ])
                
                    frames_main.append(frame)
                
                fig.add_trace(go.Scatter(), row=1, col=1)
                fig.add_trace(go.Scatter(x=bins, y=np.zeros_like(bins)), row=2, col=1)
                fig.add_trace(go.Scatter(), row=3, col=1)
                
                fig.frames = frames_main
                
                fig.update_layout(
                    height=500,
                    title=f"Global systematics behaviour {s} | {v} | {r}",
                    updatemenus=[{
                        "buttons": [{
                            "label": "Play",
                            "method": "animate",
                            "args": [None, {"frame": {"duration": 50, "redraw": True}}]
                        }]
                    }]
                )
                
                #fig.show()
                #########################################################

                pairs = [
                    (i, j)
                    for i, j in zip(groups["flux"][:-1], groups["det"][1:])
                ][:20]
                

                fig_corr = make_subplots(
                    rows=2, cols=10,
                    subplot_titles=[
                        f"{param_names[i]} vs {param_names[j]}"
                        for i, j in pairs
                    ]
                )
                

                for r in range(1, 3):
                    for c in range(1, 11):
                        fig_corr.add_trace(go.Scatter(), row=r, col=c)
                

                frames_corr = []
                
                for t in range(1, len(X_sorted)):
                
                    data = []
                
                    for (p1, p2) in pairs:
                
                        data.append(
                            go.Scatter(
                                x=X_sorted[:t, p1],
                                y=X_sorted[:t, p2],
                                mode="markers",
                                marker=dict(size=4)
                                
                            )
                        )
                
                    frames_corr.append(
                        go.Frame(
                            data=data,
                            traces=list(range(len(pairs)))
                        )
                    )
                

                fig_corr.frames = frames_corr
                

                fig_corr.update_layout(
                    height=600,
                    title=f"Parameter correlations (20 pairs) {s} | {v} | {r}",
                    showlegend=False,
                    updatemenus=[{
                        "buttons": [{
                            "label": "Play",
                            "method": "animate",
                            "args": [None, {"frame": {"duration": 50, "redraw": True}}]
                        }]
                    }]
                )
                

                #fig_corr.show()